In [1]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# BLOCO 0: CARREGAMENTO, RENOMEAÇÃO E LIMPEZA DOS DADOS BRUTOS
# ==============================================================================

print("===" * 15)
print(" Pipeline de Dados - Matriz de Red Flags")
print("===" * 15)

# 1. Definição do Caminho e Carregamento
caminho_arquivo = input("\nCole o caminho do seu CSV bruto: ").strip('"').strip("'")
print("\n⏳ Lendo a base de dados (Isso pode levar alguns segundos)...")

# Lendo o CSV. O separador é ponto e vírgula e o encoding padrão da extração foi mantido
try:
    df_bruto = pd.read_csv(caminho_arquivo, sep=';', encoding='latin1', low_memory=False)
except Exception as e:
    print(f"Erro ao carregar o arquivo: {e}")
    exit()

# 2. Dicionário de Renomeação (De/Para)
mapa_colunas = {
    'ID_ANO_LANC': 'Ano',
    'ID_MES_LANC': 'Mes',
    'ID_ORGAO': 'Cod_Orgao',
    'NO_ORGAO': 'Nome_Orgao', 
    'ID_CONTA_CONTABIL': 'Cod_Conta',
    'ID_GRUPO_DESPESA_NADE': 'Cod_GND',
    'ID_ELEMENTO_DESPESA_NADE': 'Cod_Elemento',
    'MOVIMLQUIDONOMSRCONTACONTBIL': 'Mov_Liquido'
}

# Aplicando a renomeação
df = df_bruto.rename(columns=mapa_colunas)

# 3. Filtragem de Colunas Úteis (Otimização de Memória)
# Mantemos o Nome_Orgao para rastreabilidade, mas focamos nos IDs para os cálculos
colunas_necessarias = [
    'Ano', 'Mes', 'Cod_Orgao', 'Nome_Orgao', 
    'Cod_Conta', 'Cod_GND', 'Cod_Elemento', 'Mov_Liquido'
]

# Filtra apenas as colunas que mapeamos (descartando as demais para liberar RAM)
colunas_presentes = [col for col in colunas_necessarias if col in df.columns]
df = df[colunas_presentes]

# 4. Limpeza Financeira e de Tipos
print("⚙️ Padronizando tipos de dados e limpando valores financeiros...")

# Garante que os códigos sejam tratados como texto (string) para não perder zeros à esquerda nas contas
cols_texto = ['Cod_Orgao', 'Cod_Conta', 'Cod_GND', 'Cod_Elemento']
for col in cols_texto:
    if col in df.columns:
        # Preenche vazios com '0' e converte para string
        df[col] = df[col].fillna('0').astype(str)
        # Remove eventuais casas decimais que o Pandas coloca ao ler números vazios (ex: '153046.0' -> '153046')
        df[col] = df[col].apply(lambda x: x.split('.')[0])

def limpar_valor_financeiro(valor):
    """
    Converte o padrão de exportação (ex: -576.687,41) para float matemático do Python (-576687.41)
    """
    if pd.isna(valor):
        return 0.0
    if isinstance(valor, (float, int)):
        return float(valor)
    
    valor_str = str(valor).strip()
    # Remove os pontos de milhar e troca vírgula decimal por ponto decimal
    valor_str = valor_str.replace('.', '').replace(',', '.')
    
    try:
        return float(valor_str)
    except ValueError:
        return 0.0 

# Aplica a função de limpeza na coluna de Movimento Líquido
if 'Mov_Liquido' in df.columns:
    df['Mov_Liquido'] = df['Mov_Liquido'].apply(limpar_valor_financeiro)

print(f"\n✅ Bloco 0 Concluído!")
print(f"Estrutura otimizada: {df.shape[0]} linhas e {df.shape[1]} colunas.")
print(df.head())

 Pipeline de Dados - Matriz de Red Flags

⏳ Lendo a base de dados (Isso pode levar alguns segundos)...
⚙️ Padronizando tipos de dados e limpando valores financeiros...

✅ Bloco 0 Concluído!
Estrutura otimizada: 2484563 linhas e 8 colunas.
    Ano  Mes Cod_Orgao                       Nome_Orgao  Cod_Conta Cod_GND  \
0  2023    1     26231  UNIVERSIDADE FEDERAL DE ALAGOAS  113110101      -9   
1  2023    1     26231  UNIVERSIDADE FEDERAL DE ALAGOAS  123110105      -9   
2  2023    1     26231  UNIVERSIDADE FEDERAL DE ALAGOAS  123110201      -9   
3  2023    1     26231  UNIVERSIDADE FEDERAL DE ALAGOAS  123110503      -9   
4  2023    1     26231  UNIVERSIDADE FEDERAL DE ALAGOAS  211110101      -9   

  Cod_Elemento  Mov_Liquido  
0           -9         0.00  
1           -9         0.00  
2           -9         0.00  
3           -9         0.00  
4           -9   -576687.41  


In [2]:
# %%
# ==============================================================================
# BLOCO 1: FEATURE F1 - CORRELAÇÃO ENTRE VPA E RECEITA ARRECADADA (CORRIGIDO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F1: Correlacao VPA vs Receita Arrecadada (Pura e Líquida)")
print("="*50)

import warnings
warnings.filterwarnings('ignore')

# 1. Isolar os lançamentos das Classes específicas
# Contas que começam com '4' (VPA)
df_vpa = df[df['Cod_Conta'].str.startswith('4')].copy()

# Contas que começam com '6212' (Receita Realizada/Arrecadada) + '8211' (EXECUÇÃO DA DISPONIBILIDADE DE RECURSOS)
df_rec = df[df['Cod_Conta'].str.startswith(('6212', '8211'))].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Crucial para abater estornos)
vpa_mensal = df_vpa.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPA')
rec_mensal = df_rec.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Rec')

# 3. Ajustar a Natureza (Contas Credoras no SIAFI são negativas, multiplicamos por -1)
vpa_mensal['Valor_VPA'] = vpa_mensal['Soma_VPA'] * -1
rec_mensal['Valor_Rec'] = rec_mensal['Soma_Rec'] * -1

# 4. Unir (Merge) a competência e o caixa lado a lado no mesmo mês
df_f1_mensal = pd.merge(vpa_mensal, rec_mensal, 
                        on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                        how='outer').fillna(0)

# 5. Calcular a Correlação de Pearson para cada Órgão ao longo dos 36 meses
def calcular_correlacao(group):
    # Trava de segurança para desvio padrão zero
    if group['Valor_VPA'].std() == 0 or group['Valor_Rec'].std() == 0:
        return 0.0
    return group['Valor_VPA'].corr(group['Valor_Rec'])

# Aplica a fórmula matemática
f1_resultado = df_f1_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_correlacao).reset_index(name='F1_Cor_VPA_RecArr')

# 6. Exibir os Resultados
print("✅ F1 Calculada com sucesso!")
print("\nAmostra das Correlações encontradas:")
print(f1_resultado.head(10))

# Recriando a Matriz Mestre
df_features = f1_resultado.copy()


 Calculando F1: Correlacao VPA vs Receita Arrecadada (Pura e Líquida)
✅ F1 Calculada com sucesso!

Amostra das Correlações encontradas:
  Cod_Orgao                              Nome_Orgao  F1_Cor_VPA_RecArr
0     26231         UNIVERSIDADE FEDERAL DE ALAGOAS           0.882071
1     26232           UNIVERSIDADE FEDERAL DA BAHIA           0.972744
2     26233           UNIVERSIDADE FEDERAL DO CEARA           0.955442
3     26234  UNIVERSIDADE FEDERAL DO ESPIRITO SANTO           0.966780
4     26235           UNIVERSIDADE FEDERAL DE GOIAS           0.844453
5     26236         UNIVERSIDADE FEDERAL FLUMINENSE           0.874423
6     26237    UNIVERSIDADE FEDERAL DE JUIZ DE FORA           0.245185
7     26238    UNIVERSIDADE FEDERAL DE MINAS GERAIS           0.931143
8     26239            UNIVERSIDADE FEDERAL DO PARA           0.924645
9     26240         UNIVERSIDADE FEDERAL DA PARAIBA           0.974350


In [3]:
# %%
# ==============================================================================
# BLOCO 2: FEATURE F2 - CORRELAÇÃO ENTRE VPA E CRÉDITOS A CURTO PRAZO
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F2: Correlacao VPA vs Creditos a Curto Prazo (Líquida)")
print("="*50)

# 1. Isolar os lançamentos
# VPA: Contas iniciadas em '4'
df_vpa_f2 = df[df['Cod_Conta'].str.startswith('4')].copy()

# Créditos a Curto Prazo: Contas iniciadas em '112' (Classe 1.1.2 do PCASP)
df_creditos = df[df['Cod_Conta'].str.startswith('112')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Crucial para neutralizar estornos)
vpa_mensal_f2 = df_vpa_f2.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPA')
cred_mensal = df_creditos.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Cred')

# 3. Ajustar a Natureza das Contas
# VPA (Classe 4) é credora -> Multiplicamos por -1 para ter volume positivo
vpa_mensal_f2['Valor_VPA'] = vpa_mensal_f2['Soma_VPA'] * -1

# Créditos (Classe 1.1.2) são devedores -> O saldo positivo já reflete o aumento do direito
cred_mensal['Valor_Cred'] = cred_mensal['Soma_Cred']

# 4. Unir os dados no mesmo mês
df_f2_mensal = pd.merge(vpa_mensal_f2, cred_mensal, 
                        on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                        how='outer').fillna(0)

# 5. Função de Correlação
def calcular_correlacao_f2(group):
    # Trava de segurança para desvio padrão zero
    if group['Valor_VPA'].std() == 0 or group['Valor_Cred'].std() == 0:
        return 0.0
    return group['Valor_VPA'].corr(group['Valor_Cred'])

# Aplica a fórmula matemática
f2_resultado = df_f2_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_correlacao_f2).reset_index(name='F2_Cor_VPA_Cred')

# 6. Integrar à Matriz Mestre
if 'F2_Cor_VPA_Cred' in df_features.columns:
    df_features = df_features.drop(columns=['F2_Cor_VPA_Cred'])

df_features = pd.merge(df_features, f2_resultado[['Cod_Orgao', 'F2_Cor_VPA_Cred']], 
                       on='Cod_Orgao', 
                       how='left')

print("✅ F2 Corrigida e Calculada com sucesso!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F2_Cor_VPA_Cred']].head(10))


 Calculando F2: Correlacao VPA vs Creditos a Curto Prazo (Líquida)
✅ F2 Corrigida e Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F2_Cor_VPA_Cred
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.000000
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.000000
2,26233,UNIVERSIDADE FEDERAL DO CEARA,-0.065218
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.000000
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,-0.190647
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,0.189659
6,26237,UNIVERSIDADE FEDERAL DE JUIZ DE FORA,0.000000
7,26238,UNIVERSIDADE FEDERAL DE MINAS GERAIS,-0.177312
8,26239,UNIVERSIDADE FEDERAL DO PARA,-0.110699
9,26240,UNIVERSIDADE FEDERAL DA PARAIBA,0.000000


In [4]:
# %%
# ==============================================================================
# BLOCO 3: FEATURE F3 - VOLATILIDADE DAS RECEITAS (VPA - CORRIGIDO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F3: Volatilidade das Receitas (Soma Líquida + Absoluto)")
print("="*50)

# 1. Isolar os lançamentos da Classe 4
df_vpa_f3 = df[df['Cod_Conta'].str.startswith('4')].copy()

# 2. Agrupar e Somar (Abate estornos mantendo o sinal original)
vpa_mensal_f3 = df_vpa_f3.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Mensal')

# 3. CORREÇÃO: Pegamos o valor absoluto da SOMA LÍQUIDA
vpa_mensal_f3['Valor_VPA'] = vpa_mensal_f3['Soma_Mensal'].abs()

# 4. Cálculo do CV
def calcular_cv_vpa(group):
    media = group['Valor_VPA'].mean()
    desvio = group['Valor_VPA'].std()
    if media == 0 or pd.isna(media): return 0.0
    return desvio / media

f3_resultado = vpa_mensal_f3.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_vpa).reset_index(name='F3_Volatilidade_VPA')

# 5. Merge na Matriz Mestre
if 'F3_Volatilidade_VPA' in df_features.columns:
    df_features = df_features.drop(columns=['F3_Volatilidade_VPA'])
df_features = pd.merge(df_features, f3_resultado[['Cod_Orgao', 'F3_Volatilidade_VPA']], on='Cod_Orgao', how='left').fillna(0)

print("✅ F3 Ajustada!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F3_Volatilidade_VPA']].head())


 Calculando F3: Volatilidade das Receitas (Soma Líquida + Absoluto)
✅ F3 Ajustada!


,Cod_Orgao,Nome_Orgao,F3_Volatilidade_VPA
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.199535
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.184533
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.178102
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.196227
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.198488


In [5]:
# %%
# ==============================================================================
# BLOCO 4: FEATURE F4 - VOLATILIDADE DAS DESPESAS (VPD)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F4: Volatilidade das Despesas (Soma Líquida + Absoluto)")
print("="*50)

# 1. Isolar os lançamentos da Classe 3 (VPD - Variação Patrimonial Diminutiva)
df_vpd_f4 = df[df['Cod_Conta'].str.startswith('3')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater os estornos e glosas corretamente)
vpd_mensal_f4 = df_vpd_f4.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Mensal')

# 3. Aplicar o Valor Absoluto no saldo já purificado
# A Classe 3 tem natureza devedora (positiva), mas usamos o .abs() para garantir que 
# eventuais meses com mais estornos do que despesas não quebrem o cálculo de dispersão.
vpd_mensal_f4['Valor_VPD'] = vpd_mensal_f4['Soma_Mensal'].abs()

# 4. Função do Coeficiente de Variação (CV = Desvio Padrão / Média)
def calcular_cv_vpd(group):
    media = group['Valor_VPD'].mean()
    desvio = group['Valor_VPD'].std()
    
    # Trava de segurança: se a média for zero (universidade sem despesa), o CV é zero
    if media == 0 or pd.isna(media):
        return 0.0
    
    return desvio / media

# 5. Aplica a fórmula matemática aos órgãos
f4_resultado = vpd_mensal_f4.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_vpd).reset_index(name='F4_Volatilidade_VPD')

# 6. Integrar à Matriz Mestre
# Remove a coluna antiga para evitar colunas duplicadas (ex: F4_x e F4_y) no merge
if 'F4_Volatilidade_VPD' in df_features.columns:
    df_features = df_features.drop(columns=['F4_Volatilidade_VPD'])

df_features = pd.merge(df_features, f4_resultado[['Cod_Orgao', 'F4_Volatilidade_VPD']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche vazios com zero
df_features['F4_Volatilidade_VPD'] = df_features['F4_Volatilidade_VPD'].fillna(0)

# 7. Exibir como a matriz está ficando
print("✅ F4 Corrigida e Calculada com sucesso!")
print("\nVisão atual da Matriz de Features:")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F3_Volatilidade_VPA', 'F4_Volatilidade_VPD']].head())


 Calculando F4: Volatilidade das Despesas (Soma Líquida + Absoluto)
✅ F4 Corrigida e Calculada com sucesso!

Visão atual da Matriz de Features:


,Cod_Orgao,Nome_Orgao,F3_Volatilidade_VPA,F4_Volatilidade_VPD
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.199535,0.152592
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.184533,0.198575
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.178102,0.135739
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.196227,0.240287
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.198488,0.150498


In [6]:
# %%
# ==============================================================================
# BLOCO 5: FEATURE F5 - VARIAÇÃO ENTRE PAGAMENTOS E REPASSES (CORRIGIDO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F5: Variação Pagamentos vs Repasses (Pura e Líquida)")
print("="*50)

# 1. Isolar os lançamentos
# Pagamentos: Contas iniciadas em '62213' (Execução da Despesa Paga)
df_pag = df[df['Cod_Conta'].str.startswith('62213')].copy()

# Repasses: Contas iniciadas em '8211' (Cota/Repasse/Subrepasse Recebido)
df_rep = df[df['Cod_Conta'].str.startswith('8211')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater os estornos corretamente)
pag_mensal = df_pag.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Pag')
rep_mensal = df_rep.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Rep')

# 3. Ajustar a Natureza das Contas
# Ambas as contas (62213 e 8211) possuem natureza CREDORA no balancete do SIAFI.

pag_mensal['Valor_Pag'] = pag_mensal['Soma_Pag'].abs()
rep_mensal['Valor_Rep'] = rep_mensal['Soma_Rep'].abs()

# 4. Unir os dados no mesmo mês
df_f5_mensal = pd.merge(pag_mensal, rep_mensal, 
                        on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                        how='outer').fillna(0)

# 5. Criar a Razão Mensal (Pagamentos / Repasses)
# np.where garante que se o repasse no mês for zero, a razão seja zero (evitando erro de infinitos)
df_f5_mensal['Razao_Pag_Rep'] = np.where(df_f5_mensal['Valor_Rep'] == 0, 
                                         0.0, 
                                         df_f5_mensal['Valor_Pag'] / df_f5_mensal['Valor_Rep'])

# 6. Função do Coeficiente de Variação (CV = Desvio Padrão / Média) da Razão
def calcular_cv_f5(group):
    media = group['Razao_Pag_Rep'].mean()
    desvio = group['Razao_Pag_Rep'].std()
    
    # Trava de segurança
    if media == 0 or pd.isna(media):
        return 0.0
    return desvio / media

# 7. Aplica a fórmula matemática
f5_resultado = df_f5_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f5).reset_index(name='F5_Var_Pag_Repasse')

# 8. Integrar à Matriz Mestre
# Remove a coluna antiga se ela já existir para não duplicar no merge
if 'F5_Var_Pag_Repasse' in df_features.columns:
    df_features = df_features.drop(columns=['F5_Var_Pag_Repasse'])

df_features = pd.merge(df_features, f5_resultado[['Cod_Orgao', 'F5_Var_Pag_Repasse']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche possíveis vazios com zero
df_features['F5_Var_Pag_Repasse'] = df_features['F5_Var_Pag_Repasse'].fillna(0)

print("✅ F5 Calculada com sucesso!")
print("\nVisão atual da Matriz de Features:")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F1_Cor_VPA_RecArr', 'F5_Var_Pag_Repasse']].head())


 Calculando F5: Variação Pagamentos vs Repasses (Pura e Líquida)
✅ F5 Calculada com sucesso!

Visão atual da Matriz de Features:


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F5_Var_Pag_Repasse
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,2.731987
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,2.627832
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,2.671709
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,2.692641
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,2.602846


In [7]:
# %%
# ==============================================================================
# BLOCO 6: FEATURE F6 - VOLATILIDADE DO RESULTADO PATRIMONIAL SOBRE O PL (ROE PÚBLICO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F6: Volatilidade do RPE sobre o PL (F6_Var_RPE_PL)")
print("="*50)

# 1. Isolar os lançamentos
# VPA (Classe 4) e VPD (Classe 3) para calcular o Resultado Patrimonial
df_vpa_f6 = df[df['Cod_Conta'].str.startswith('4')].copy()
df_vpd_f6 = df[df['Cod_Conta'].str.startswith('3')].copy()

# Patrimônio Líquido (Classe 2.3)
df_pl = df[df['Cod_Conta'].str.startswith('23')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Abate estornos e ajustes)
vpa_mensal_f6 = df_vpa_f6.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPA')
vpd_mensal_f6 = df_vpd_f6.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPD')
pl_mensal = df_pl.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_PL')

# 3. Aplicar Valor Absoluto APÓS a soma (Extrai o volume real compensado)
vpa_mensal_f6['Valor_VPA'] = vpa_mensal_f6['Soma_VPA'].abs()
vpd_mensal_f6['Valor_VPD'] = vpd_mensal_f6['Soma_VPD'].abs()
# Trabalhando com a magnitude bruta da variação do PL no mês
pl_mensal['Valor_PL'] = pl_mensal['Soma_PL'].abs()

# 4. Unir os dados no mesmo mês
# Primeiro unimos VPA e VPD
df_resultado_mensal = pd.merge(vpa_mensal_f6, vpd_mensal_f6, on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], how='outer').fillna(0)
# Depois unimos o PL
df_f6_mensal = pd.merge(df_resultado_mensal, pl_mensal, on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], how='outer').fillna(0)

# 5. Calcular o Resultado Patrimonial (RPE = VPA - VPD) e a Razão (ROE = RPE / PL)
df_f6_mensal['RPE'] = df_f6_mensal['Valor_VPA'] - df_f6_mensal['Valor_VPD']

# np.where garante que se não houver movimentação no PL no mês, a razão seja zero para evitar erro
df_f6_mensal['Razao_ROE'] = np.where(df_f6_mensal['Valor_PL'] == 0, 
                                     0.0, 
                                     df_f6_mensal['RPE'] / df_f6_mensal['Valor_PL'])

# 6. Função do Coeficiente de Variação (CV = Desvio Padrão / Média) da Razão
def calcular_cv_f6(group):
    media = group['Razao_ROE'].mean()
    desvio = group['Razao_ROE'].std()
    
    # Trava de segurança para evitar divisão por zero
    if media == 0 or pd.isna(media):
        return 0.0
    return abs(desvio / media) # Absoluto para garantir que a volatilidade seja uma magnitude positiva

# 7. Aplica a fórmula matemática
f6_resultado = df_f6_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f6).reset_index(name='F6_Var_RPE_PL')

# 8. Integrar à Matriz Mestre
df_features = pd.merge(df_features, f6_resultado[['Cod_Orgao', 'F6_Var_RPE_PL']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche possíveis vazios com zero
df_features['F6_Var_RPE_PL'] = df_features['F6_Var_RPE_PL'].fillna(0)

print("✅ F6 Calculada com sucesso!")
display(df_features.head(10))


 Calculando F6: Volatilidade do RPE sobre o PL (F6_Var_RPE_PL)
✅ F6 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,0.874423,0.189659,0.185052,0.176741,0.171454,6.114074
6,26237,UNIVERSIDADE FEDERAL DE JUIZ DE FORA,0.245185,0.000000,0.647403,0.140351,2.349077,5.880085
7,26238,UNIVERSIDADE FEDERAL DE MINAS GERAIS,0.931143,-0.177312,0.179566,0.159603,2.598110,9.574437
8,26239,UNIVERSIDADE FEDERAL DO PARA,0.924645,-0.110699,0.172736,0.188800,1.595068,6.000000
9,26240,UNIVERSIDADE FEDERAL DA PARAIBA,0.974350,0.000000,0.167452,0.136707,2.399571,6.215825


In [8]:
# %%
# ==============================================================================
# BLOCO 7: FEATURE F7 - QUANTIDADE DE OCORRÊNCIAS DE DÉFICIT FINANCEIRO
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F7: Quantidade de Déficits Financeiros (F7_Qtd_Deficit_Fin)")
print("="*50)

# 1. Isolar os lançamentos de Entrada e Saída de Caixa
# Entradas: Receitas Arrecadadas (6212) + Cota/Repasse/Subrepasse Recebido (8211)
df_entradas = df[df['Cod_Conta'].str.startswith(('6212', '8211'))].copy()

# Saídas: Despesas Pagas (62213)
df_saidas = df[df['Cod_Conta'].str.startswith('62213')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater estornos financeiras e ordens bancárias canceladas)
entradas_mensal = df_entradas.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Entrada')
saidas_mensal = df_saidas.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Saida')

# 3. Aplicar o valor absoluto APÓS a compensação dos estornos
# Como todas essas contas (6212, 8211 e 62213) possuem natureza credora,
# o .abs() garante que estamos comparando o volume real de dinheiro movimentado.
entradas_mensal['Valor_Entrada'] = entradas_mensal['Soma_Entrada'].abs()
saidas_mensal['Valor_Saida'] = saidas_mensal['Soma_Saida'].abs()

# 4. Unir os dados no mesmo mês para confrontar o saldo
df_f7_mensal = pd.merge(entradas_mensal, saidas_mensal, 
                        on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                        how='outer').fillna(0)

# 5. Criar a Flag de Déficit (1 se Saída > Entrada, 0 caso contrário)
df_f7_mensal['Is_Deficit'] = np.where(df_f7_mensal['Valor_Saida'] > df_f7_mensal['Valor_Entrada'], 1, 0)

# 6. Aplicar a Métrica (Somar os meses com déficit para cada universidade)
f7_resultado = df_f7_mensal.groupby(['Cod_Orgao', 'Nome_Orgao'])['Is_Deficit'].sum().reset_index(name='F7_Qtd_Deficit_Fin')

# 7. Atualizar a Matriz Mestre (Se a coluna já existir de um erro anterior, nós a dropamos antes)
if 'F7_Qtd_Deficit_Fin' in df_features.columns:
    df_features = df_features.drop(columns=['F7_Qtd_Deficit_Fin'])

df_features = pd.merge(df_features, f7_resultado[['Cod_Orgao', 'F7_Qtd_Deficit_Fin']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F7_Qtd_Deficit_Fin'] = df_features['F7_Qtd_Deficit_Fin'].fillna(0).astype(int)

print("✅ F7 Calculada com sucesso!")
display(df_features.head())


 Calculando F7: Quantidade de Déficits Financeiros (F7_Qtd_Deficit_Fin)
✅ F7 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4


In [9]:
# %%
# ==============================================================================
# BLOCO 8: FEATURE F8 - VOLATILIDADE DA MARGEM PATRIMONIAL (CORRIGIDO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F8: Volatilidade da Margem Patrimonial (Saldos Líquidos Ajustados)")
print("="*50)

# 1. Isolar os lançamentos (Puramente Patrimoniais)
# Receitas (VPA - Classe 4)
df_vpa_f8 = df[df['Cod_Conta'].str.startswith('4')].copy()
# Consumo (VPD - Classe 3)
df_vpd_f8 = df[df['Cod_Conta'].str.startswith('3')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater estornos internamente)
vpa_mensal_f8 = df_vpa_f8.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPA')
vpd_mensal_f8 = df_vpd_f8.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPD')

# 3. Aplicar o Padrão Ouro: Valor absoluto APÓS a compensação dos estornos
# Isso garante que teremos a magnitude positiva do volume de receitas e despesas
vpa_mensal_f8['Valor_VPA'] = vpa_mensal_f8['Soma_VPA'].abs()
vpd_mensal_f8['Valor_VPD'] = vpd_mensal_f8['Soma_VPD'].abs()

# 4. Unir os dados no mesmo mês
df_f8_mensal = pd.merge(vpa_mensal_f8, vpd_mensal_f8, 
                        on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                        how='outer').fillna(0)

# 5. Calcular o RPE e a Margem Patrimonial
df_f8_mensal['RPE'] = df_f8_mensal['Valor_VPA'] - df_f8_mensal['Valor_VPD']

# A Margem é RPE / VPA. Usamos np.where para evitar divisão por zero se a VPA for nula no mês.
df_f8_mensal['Margem_Patrimonial'] = np.where(df_f8_mensal['Valor_VPA'] == 0, 
                                              0.0, 
                                              df_f8_mensal['RPE'] / df_f8_mensal['Valor_VPA'])

# 6. Função do Coeficiente de Variação (CV) da Margem Patrimonial
def calcular_cv_f8(group):
    media = group['Margem_Patrimonial'].mean()
    desvio = group['Margem_Patrimonial'].std()
    
    # Trava de segurança
    if media == 0 or pd.isna(media):
        return 0.0
    
    # Retornamos o valor absoluto do CV para garantir a escala positiva de volatilidade
    return abs(desvio / media)

# 7. Aplica a fórmula matemática
f8_resultado = df_f8_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f8).reset_index(name='F8_Var_Margem_Pat')

# 8. Integrar à Matriz Mestre
if 'F8_Var_Margem_Pat' in df_features.columns:
    df_features = df_features.drop(columns=['F8_Var_Margem_Pat'])
if 'F8_Desv_MB' in df_features.columns:
    df_features = df_features.drop(columns=['F8_Desv_MB'])

df_features = pd.merge(df_features, f8_resultado[['Cod_Orgao', 'F8_Var_Margem_Pat']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche vazios com zero
df_features['F8_Var_Margem_Pat'] = df_features['F8_Var_Margem_Pat'].fillna(0)

print("✅ F8 Ajustada e Calculada com sucesso!")
display(df_features.head())



 Calculando F8: Volatilidade da Margem Patrimonial (Saldos Líquidos Ajustados)
✅ F8 Ajustada e Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3,2.191451
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3,4.889526
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4,5.181242
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4,20.780069
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4,2.853935


In [10]:
# %%
# ==============================================================================
# BLOCO 9: FEATURE F9 - TENDÊNCIA DO RESULTADO PATRIMONIAL (CORRIGIDO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F9: Tendência do Resultado Patrimonial")
print("="*50)

import numpy as np # Garantindo que a biblioteca está carregada

# 1. Isolar os lançamentos (VPA e VPD)
df_vpa_f9 = df[df['Cod_Conta'].str.startswith('4')].copy()
df_vpd_f9 = df[df['Cod_Conta'].str.startswith('3')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater estornos internamente)
vpa_mensal_f9 = df_vpa_f9.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPA')
vpd_mensal_f9 = df_vpd_f9.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPD')

# 3. Aplicar o Padrão Ouro: Valor absoluto APÓS a compensação
vpa_mensal_f9['Valor_VPA'] = vpa_mensal_f9['Soma_VPA'].abs()
vpd_mensal_f9['Valor_VPD'] = vpd_mensal_f9['Soma_VPD'].abs()

# 4. Unir os dados no mesmo mês
df_f9_mensal = pd.merge(vpa_mensal_f9, vpd_mensal_f9, 
                        on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                        how='outer').fillna(0)

# 5. Calcular o Resultado Patrimonial (RPE = VPA - VPD)
df_f9_mensal['RPE'] = df_f9_mensal['Valor_VPA'] - df_f9_mensal['Valor_VPD']

# 6. Ordenar cronologicamente para garantir a linha do tempo correta da regressão
df_f9_mensal = df_f9_mensal.sort_values(by=['Cod_Orgao', 'Ano', 'Mes'])

# 7. Função do Coeficiente Angular usando numpy.polyfit
def calcular_tendencia_f9(group):
    # O eixo Y são os valores reais e limpos do RPE
    y = group['RPE'].values
    n = len(y)
    
    # Trava de segurança: precisamos de pelo menos 2 pontos
    if n < 2:
        return 0.0
    
    # O eixo X é a linha do tempo
    x = np.arange(1, n + 1)
    
    # O polyfit extrai o coeficiente angular (inclinação)
    coeficiente_angular, _ = np.polyfit(x, y, 1)
    
    return coeficiente_angular

# 8. Aplica a fórmula matemática
f9_resultado = df_f9_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_tendencia_f9).reset_index(name='F9_Ang_RPE')

# 9. Integrar à Matriz Mestre
if 'F9_Ang_RPE' in df_features.columns:
    df_features = df_features.drop(columns=['F9_Ang_RPE'])

df_features = pd.merge(df_features, f9_resultado[['Cod_Orgao', 'F9_Ang_RPE']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche vazios com zero
df_features['F9_Ang_RPE'] = df_features['F9_Ang_RPE'].fillna(0)

print("✅ F9 Calculada com sucesso!")
display(df_features.head())



 Calculando F9: Tendência do Resultado Patrimonial
✅ F9 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat,F9_Ang_RPE
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3,2.191451,18067.915499
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3,4.889526,43414.701403
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4,5.181242,99046.096090
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4,20.780069,-122363.979730
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4,2.853935,-1888.806882


In [11]:
# %%
# ==============================================================================
# BLOCO 10: FEATURE F10 - VOLATILIDADE DA NECESSIDADE DE FINANCIAMENTO (NFCP)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F10: Volatilidade da NFCP")
print("="*50)

from functools import reduce
import numpy as np
import pandas as pd

# 1. Isolar as contas do PCASP detalhadas na metodologia
# Bens de Rápida Conversão: Créditos (112) e Estoques (113)
df_112 = df[df['Cod_Conta'].str.startswith('112')].copy()
df_113 = df[df['Cod_Conta'].str.startswith('113')].copy()

# Obrigações Imediatas: Fornecedores e Contas a Pagar a curto prazo (213)
df_213 = df[df['Cod_Conta'].str.startswith('213')].copy()

# Denominador (O Tamanho da Universidade): Ativo Total (Classe 1)
df_1 = df[df['Cod_Conta'].str.startswith('1')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (A vacina contra os estornos intra-mês)
mensal_112 = df_112.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_112')
mensal_113 = df_113.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_113')
mensal_213 = df_213.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_213')
mensal_1 = df_1.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_1')

# 3. Aplicar o Padrão Ouro: Valor absoluto APÓS a compensação
# Extraímos a magnitude real de movimentação de cada conta no mês
mensal_112['Valor_112'] = mensal_112['Soma_112'].abs()
mensal_113['Valor_113'] = mensal_113['Soma_113'].abs()
mensal_213['Valor_213'] = mensal_213['Soma_213'].abs()
mensal_1['Valor_1'] = mensal_1['Soma_1'].abs()

# 4. Mesclar todos os dataframes no mesmo eixo de tempo
# O 'reduce' aplica o merge sequencialmente em toda a lista, garantindo que nenhum mês fique de fora
dataframes = [
    mensal_112[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'Valor_112']],
    mensal_113[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'Valor_113']],
    mensal_213[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'Valor_213']],
    mensal_1[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'Valor_1']]
]

df_f10_mensal = reduce(lambda left, right: pd.merge(left, right, on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], how='outer'), dataframes).fillna(0)

# 5. Calcular a Balança (NFCP) e a Proporção sobre o Ativo
df_f10_mensal['NFCP'] = (df_f10_mensal['Valor_112'] + df_f10_mensal['Valor_113']) - df_f10_mensal['Valor_213']

# Evitar divisão por zero caso haja algum mês sem movimentação no Ativo
df_f10_mensal['Razao_NFCP'] = np.where(df_f10_mensal['Valor_1'] == 0, 
                                       0.0, 
                                       df_f10_mensal['NFCP'] / df_f10_mensal['Valor_1'])

# 6. Função do Coeficiente de Variação (CV) da Razão NFCP
def calcular_cv_f10(group):
    media = group['Razao_NFCP'].mean()
    desvio = group['Razao_NFCP'].std()
    
    if media == 0 or pd.isna(media):
        return 0.0
    
    # Retornamos o absoluto para garantir uma escala positiva de volatilidade
    return abs(desvio / media)

# 7. Aplica a fórmula matemática
f10_resultado = df_f10_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f10).reset_index(name='F10_Volatilidade_NFCP')

# 8. Integrar à Matriz Mestre
if 'F10_Volatilidade_NFCP' in df_features.columns:
    df_features = df_features.drop(columns=['F10_Volatilidade_NFCP'])

df_features = pd.merge(df_features, f10_resultado[['Cod_Orgao', 'F10_Volatilidade_NFCP']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche vazios com zero
df_features['F10_Volatilidade_NFCP'] = df_features['F10_Volatilidade_NFCP'].fillna(0)

print("✅ F10 Calculada com sucesso!")
display(df_features.head())



 Calculando F10: Volatilidade da NFCP
✅ F10 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat,F9_Ang_RPE,F10_Volatilidade_NFCP
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3,2.191451,18067.915499,3.935866
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3,4.889526,43414.701403,6.700012
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4,5.181242,99046.096090,5.517778
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4,20.780069,-122363.979730,72.233011
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4,2.853935,-1888.806882,5.596572


In [12]:
# %%
# ==============================================================================
# BLOCO 11: FEATURE F11 - TENDÊNCIA DO PASSIVO CIRCULANTE SOBRE O ATIVO
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F11: Tendência do Endividamento Curto Prazo")
print("="*50)

import numpy as np
import pandas as pd

# 1. Isolar os lançamentos (Passivo Circulante e Ativo Total)
# Passivo Circulante (Classe 2.1)
df_passivo_f11 = df[df['Cod_Conta'].str.startswith('21')].copy()
# Ativo Total (Classe 1)
df_ativo_f11 = df[df['Cod_Conta'].str.startswith('1')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater estornos e reclassificações)
passivo_mensal = df_passivo_f11.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Passivo')
ativo_mensal = df_ativo_f11.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Ativo')

# 3. Aplicar o Padrão Ouro: Valor absoluto APÓS a compensação
passivo_mensal['Valor_Passivo'] = passivo_mensal['Soma_Passivo'].abs()
ativo_mensal['Valor_Ativo'] = ativo_mensal['Soma_Ativo'].abs()

# 4. Unir os dados no mesmo eixo de tempo
df_f11_mensal = pd.merge(passivo_mensal, ativo_mensal, 
                         on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                         how='outer').fillna(0)

# 5. Calcular a Proporção: Passivo Circulante / Ativo Total
# np.where evita erro de divisão por zero caso o Ativo seja nulo em algum mês atípico
df_f11_mensal['Razao_Passivo_Ativo'] = np.where(df_f11_mensal['Valor_Ativo'] == 0, 
                                                0.0, 
                                                df_f11_mensal['Valor_Passivo'] / df_f11_mensal['Valor_Ativo'])

# 6. Ordenar cronologicamente para garantir que a linha do tempo (Eixo X) esteja correta
df_f11_mensal = df_f11_mensal.sort_values(by=['Cod_Orgao', 'Ano', 'Mes'])

# 7. Função do Coeficiente Angular da Proporção (Tendência Linear)
def calcular_tendencia_f11(group):
    # O eixo Y agora é a proporção (um número percentual/decimal, ex: 0.05)
    y = group['Razao_Passivo_Ativo'].values
    n = len(y)
    
    # Trava de segurança: precisamos de pelo menos 2 pontos para traçar a reta
    if n < 2:
        return 0.0
    
    # O eixo X é a linha do tempo (1 a 36 meses)
    x = np.arange(1, n + 1)
    
    # Extrai apenas a inclinação da reta (slope)
    coeficiente_angular, _ = np.polyfit(x, y, 1)
    
    return coeficiente_angular

# 8. Aplica a fórmula matemática
f11_resultado = df_f11_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_tendencia_f11).reset_index(name='F11_Ang_Passivo')

# 9. Integrar à Matriz Mestre
if 'F11_Ang_Passivo' in df_features.columns:
    df_features = df_features.drop(columns=['F11_Ang_Passivo'])

df_features = pd.merge(df_features, f11_resultado[['Cod_Orgao', 'F11_Ang_Passivo']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche vazios com zero
df_features['F11_Ang_Passivo'] = df_features['F11_Ang_Passivo'].fillna(0)

print("✅ F11 Calculada com sucesso!")
display(df_features.head())


 Calculando F11: Tendência do Endividamento Curto Prazo
✅ F11 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat,F9_Ang_RPE,F10_Volatilidade_NFCP,F11_Ang_Passivo
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3,2.191451,18067.915499,3.935866,0.035900
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3,4.889526,43414.701403,6.700012,-0.011133
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4,5.181242,99046.096090,5.517778,-0.436990
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4,20.780069,-122363.979730,72.233011,0.009651
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4,2.853935,-1888.806882,5.596572,0.013369


In [13]:
# %%
# ==============================================================================
# BLOCO 12: FEATURE F12 - CORRELAÇÃO ENTRE FCO E RPE (CORRIGIDO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F12: Correlação RPE x Fluxo de Caixa Operacional")
print("="*50)

from functools import reduce
import pandas as pd

# 1. Isolar as contas Patrimoniais e Financeiras
df_vpa = df[df['Cod_Conta'].str.startswith('4')].copy()
df_vpd = df[df['Cod_Conta'].str.startswith('3')].copy()
df_entradas = df[df['Cod_Conta'].str.startswith(('6212', '8211'))].copy()
df_saidas = df[df['Cod_Conta'].str.startswith('62213')].copy()

# 2. Agrupar e Somar (Mantendo a integridade dos sinais originais da sua base)
vpa_m = df_vpa.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='VPA_Mensal')
vpd_m = df_vpd.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='VPD_Mensal')
ent_m = df_entradas.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Entradas_Mensal')
sai_m = df_saidas.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Saidas_Mensal')

# 3. Mesclar todos os dataframes no mesmo eixo de tempo
dataframes_f12 = [
    vpa_m[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'VPA_Mensal']],
    vpd_m[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'VPD_Mensal']],
    ent_m[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'Entradas_Mensal']],
    sai_m[['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes', 'Saidas_Mensal']]
]

df_f12_mensal = reduce(lambda left, right: pd.merge(left, right, on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], how='outer'), dataframes_f12).fillna(0)

# 4. Calcular os Resultados (A Matemática Direta)
df_f12_mensal['RPE_Mensal'] = df_f12_mensal['VPA_Mensal'] - df_f12_mensal['VPD_Mensal']
df_f12_mensal['FCO_Mensal'] = df_f12_mensal['Entradas_Mensal'] - df_f12_mensal['Saidas_Mensal']

# 5. Função da Correlação de Pearson
def calcular_correlacao_f12(group):
    # Trava contra erro matemático se uma das linhas for estática
    if group['RPE_Mensal'].std() == 0 or group['FCO_Mensal'].std() == 0:
        return 0.0
    return group['RPE_Mensal'].corr(group['FCO_Mensal'])

# 6. Aplica a fórmula aos órgãos
f12_resultado = df_f12_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_correlacao_f12).reset_index(name='F12_Cor_FCO_RPE')

# 7. Integrar à Matriz Mestre
if 'F12_Cor_FCO_RPE' in df_features.columns:
    df_features = df_features.drop(columns=['F12_Cor_FCO_RPE'])

df_features = pd.merge(df_features, f12_resultado[['Cod_Orgao', 'F12_Cor_FCO_RPE']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F12_Cor_FCO_RPE'] = df_features['F12_Cor_FCO_RPE'].fillna(0)

print("✅ F12 Calculada com sucesso!")
display(df_features.head())


 Calculando F12: Correlação RPE x Fluxo de Caixa Operacional
✅ F12 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat,F9_Ang_RPE,F10_Volatilidade_NFCP,F11_Ang_Passivo,F12_Cor_FCO_RPE
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3,2.191451,18067.915499,3.935866,0.035900,0.078397
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3,4.889526,43414.701403,6.700012,-0.011133,-0.047558
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4,5.181242,99046.096090,5.517778,-0.436990,-0.127917
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4,20.780069,-122363.979730,72.233011,0.009651,-0.021178
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4,2.853935,-1888.806882,5.596572,0.013369,0.204565


In [14]:
# %%
# ==============================================================================
# BLOCO 13: FEATURE F13 - VOLATILIDADE DA PROPORÇÃO INVESTIMENTO / CUSTEIO
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F13: Volatilidade Investimento/Custeio (Correção PCASP e GND)")
print("="*50)

import numpy as np
import pandas as pd

# 1. O Cruzamento Cirúrgico: Conta de Pagamento (6221304) + GND (3 ou 4)
# Filtra: Conta começa com 6221304 (Crédito Empenhado Liquidado Pago) E GND é 3 (Custeio)
df_custeio = df[(df['Cod_Conta'].str.startswith('6221304')) & (df['Cod_GND'] == '3')].copy()

# Filtra: Conta começa com 6221304 (Crédito Empenhado Liquidado Pago) E GND é 4 (Investimento)
df_investimento = df[(df['Cod_Conta'].str.startswith('6221304')) & (df['Cod_GND'] == '4')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater estornos de pagamento)
custeio_m = df_custeio.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Custeio')
invest_m = df_investimento.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Invest')

# 3. Aplicar o Padrão Ouro: Valor absoluto APÓS a compensação
custeio_m['Valor_Custeio'] = custeio_m['Soma_Custeio'].abs()
invest_m['Valor_Invest'] = invest_m['Soma_Invest'].abs()

# 4. Unir os dados no mesmo eixo de tempo
df_f13_mensal = pd.merge(custeio_m, invest_m, 
                         on=['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'], 
                         how='outer').fillna(0)

# 5. Calcular a Proporção: Investimento / Custeio
df_f13_mensal['Razao_Inv_Custeio'] = np.where(df_f13_mensal['Valor_Custeio'] == 0, 
                                              0.0, 
                                              df_f13_mensal['Valor_Invest'] / df_f13_mensal['Valor_Custeio'])

# 6. Função do Coeficiente de Variação (CV)
def calcular_cv_f13(group):
    media = group['Razao_Inv_Custeio'].mean()
    desvio = group['Razao_Inv_Custeio'].std()
    
    if media == 0 or pd.isna(media):
        return 0.0
    
    return abs(desvio / media)

# 7. Aplica a fórmula matemática
f13_resultado = df_f13_mensal.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f13).reset_index(name='F13_Var_Inv_Custeio')

# 8. Integrar à Matriz Mestre
if 'F13_Var_Inv_Custeio' in df_features.columns:
    df_features = df_features.drop(columns=['F13_Var_Inv_Custeio'])

df_features = pd.merge(df_features, f13_resultado[['Cod_Orgao', 'F13_Var_Inv_Custeio']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F13_Var_Inv_Custeio'] = df_features['F13_Var_Inv_Custeio'].fillna(0)

print("✅ F13 Calculada com extrema precisão contábil!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F13_Var_Inv_Custeio']].head())


 Calculando F13: Volatilidade Investimento/Custeio (Correção PCASP e GND)
✅ F13 Calculada com extrema precisão contábil!


,Cod_Orgao,Nome_Orgao,F13_Var_Inv_Custeio
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,2.072983
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,1.332171
2,26233,UNIVERSIDADE FEDERAL DO CEARA,1.406565
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,2.561493
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,1.072200


In [15]:
# %%
# ==============================================================================
# BLOCO 14: FEATURE F14 - VOLATILIDADE DA DISPONIBILIDADE DE CAIXA (CLASSE 1.1.1)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F14: Volatilidade da Disponibilidade de Caixa (Empoçamento)")
print("="*50)

import numpy as np
import pandas as pd

# 1. Filtro Cirúrgico: Classe 1.1.1 (Caixa e Equivalentes de Caixa / Limite de Saque)
df_caixa = df[df['Cod_Conta'].str.startswith('111')].copy()

# 2. Agrupamento mensal compensatório (Regra de Ouro: mantendo o sinal para abater estornos)
caixa_m = df_caixa.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Caixa')

# 3. Extraindo a magnitude financeira da movimentação (Regra de Ouro: .abs())
caixa_m['Valor_Caixa'] = caixa_m['Soma_Caixa'].abs()

# 4. Função do Coeficiente de Variação (CV)
def calcular_cv_f14(group):
    media = group['Valor_Caixa'].mean()
    desvio = group['Valor_Caixa'].std()
    
    if media == 0 or pd.isna(media):
        return 0.0
    
    return abs(desvio / media)

# 5. Aplicação da métrica matemática por Órgão
f14_resultado = caixa_m.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f14).reset_index(name='F14_Var_Disp_Caixa')

# 6. Integração com a Matriz Mestre
if 'F14_Var_Disp_Caixa' in df_features.columns:
    df_features = df_features.drop(columns=['F14_Var_Disp_Caixa'])

df_features = pd.merge(df_features, f14_resultado[['Cod_Orgao', 'F14_Var_Disp_Caixa']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F14_Var_Disp_Caixa'] = df_features['F14_Var_Disp_Caixa'].fillna(0)

print("✅ F14 Calculada: Rastreamento de empoçamento de recursos concluído!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F14_Var_Disp_Caixa']].head())


 Calculando F14: Volatilidade da Disponibilidade de Caixa (Empoçamento)
✅ F14 Calculada: Rastreamento de empoçamento de recursos concluído!


,Cod_Orgao,Nome_Orgao,F14_Var_Disp_Caixa
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.879982
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.917868
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.968635
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966231
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.989601


In [16]:
# %%
# ==============================================================================
# BLOCO 15: FEATURE F15 - VOLATILIDADE DE CRÉDITOS (CLASSE 1.1.2)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F15: Volatilidade de Créditos (Magnitude Mensal)")
print("="*50)

import numpy as np
import pandas as pd

# 1. Filtro: Classe 1.1.2 (Créditos a Curto Prazo)
df_creditos = df[df['Cod_Conta'].str.startswith('112')].copy()

# 2. Agrupamento mensal compensatório (Soma preservando o sinal)
creditos_m = df_creditos.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Creditos')

# 3. Aplicação do Valor Absoluto (Regra de Ouro: magnitude após compensação)
creditos_m['Valor_Creditos'] = creditos_m['Soma_Creditos'].abs()

# 4. Função do Coeficiente de Variação (CV)
def calcular_cv_f15(group):
    # Calculamos o CV sobre a coluna de valores absolutos
    media = group['Valor_Creditos'].mean()
    desvio = group['Valor_Creditos'].std()
    
    if media == 0 or pd.isna(media):
        return 0.0
    
    return abs(desvio / media)

# 5. Aplicação da métrica por Órgão
f15_resultado = creditos_m.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f15).reset_index(name='F15_Var_Creditos')

# 6. Integração à Matriz Mestre
if 'F15_Var_Creditos' in df_features.columns:
    df_features = df_features.drop(columns=['F15_Var_Creditos'])

df_features = pd.merge(df_features, f15_resultado[['Cod_Orgao', 'F15_Var_Creditos']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F15_Var_Creditos'] = df_features['F15_Var_Creditos'].fillna(0)

print("✅ F15 Calculada com a Regra de Ouro aplicada!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F15_Var_Creditos']].head(10))


 Calculando F15: Volatilidade de Créditos (Magnitude Mensal)
✅ F15 Calculada com a Regra de Ouro aplicada!


,Cod_Orgao,Nome_Orgao,F15_Var_Creditos
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.000000
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.000000
2,26233,UNIVERSIDADE FEDERAL DO CEARA,2.379465
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.000000
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,1.391732
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,2.562965
6,26237,UNIVERSIDADE FEDERAL DE JUIZ DE FORA,0.000000
7,26238,UNIVERSIDADE FEDERAL DE MINAS GERAIS,2.335497
8,26239,UNIVERSIDADE FEDERAL DO PARA,6.000000
9,26240,UNIVERSIDADE FEDERAL DA PARAIBA,0.000000


In [17]:
# %%
# ==============================================================================
# BLOCO 16: FEATURE F16 - TENDÊNCIA DAS VARIAÇÕES PATRIMONIAIS DIMINUTIVAS
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F16: Tendência da VPD (Regressão Linear)")
print("="*50)

import numpy as np

# 1. Isolar os lançamentos de VPD (Classe 3)
df_vpd_f16 = df[df['Cod_Conta'].str.startswith('3')].copy()

# 2. Agrupar e Somar MANTENDO O SINAL (Para abater estornos internamente)
vpd_mensal_f16 = df_vpd_f16.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPD')

# 3. Aplicar o Padrão Ouro: Valor absoluto APÓS a compensação mensal
vpd_mensal_f16['Valor_VPD'] = vpd_mensal_f16['Soma_VPD'].abs()

# 4. Ordenar cronologicamente (Crítico para a fidedignidade da reta de tendência)
vpd_mensal_f16 = vpd_mensal_f16.sort_values(by=['Cod_Orgao', 'Ano', 'Mes'])

# 5. Função do Coeficiente Angular (Beta) seguindo o padrão da F9
def calcular_tendencia_f16(group):
    y = group['Valor_VPD'].values
    n = len(y)
    
    # Trava de segurança: precisamos de pelo menos 2 pontos para traçar uma reta
    if n < 2:
        return 0.0
    
    # Eixo X: Linha do tempo de 1 a n meses
    x = np.arange(1, n + 1)
    
    # Extração do coeficiente angular (slope)
    coeficiente_angular, _ = np.polyfit(x, y, 1)
    
    return coeficiente_angular

# 6. Aplica a métrica por Órgão
f16_resultado = vpd_mensal_f16.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_tendencia_f16).reset_index(name='F16_Ang_VPD')

# 7. Integrar à Matriz Mestre (Limpando versões anteriores se houver)
if 'F16_Ang_VPD' in df_features.columns:
    df_features = df_features.drop(columns=['F16_Ang_VPD'])

df_features = pd.merge(df_features, f16_resultado[['Cod_Orgao', 'F16_Ang_VPD']], 
                       on='Cod_Orgao', 
                       how='left')

# Preenche vazios (universidades sem despesa registrada, o que já seria uma anomalia) com zero
df_features['F16_Ang_VPD'] = df_features['F16_Ang_VPD'].fillna(0)

print("✅ F16 Calculada: Espelhamento metodológico com a F9 concluído!")
display(df_features.head(10))


 Calculando F16: Tendência da VPD (Regressão Linear)
✅ F16 Calculada: Espelhamento metodológico com a F9 concluído!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat,F9_Ang_RPE,F10_Volatilidade_NFCP,F11_Ang_Passivo,F12_Cor_FCO_RPE,F13_Var_Inv_Custeio,F14_Var_Disp_Caixa,F15_Var_Creditos,F16_Ang_VPD
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.882071,0.000000,0.199535,0.152592,2.731987,4.601869,3,2.191451,1.806792e+04,3.935866,0.035900,0.078397,2.072983,0.879982,0.000000,1.364657e+06
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.972744,0.000000,0.184533,0.198575,2.627832,6.266523,3,4.889526,4.341470e+04,6.700012,-0.011133,-0.047558,1.332171,0.917868,0.000000,2.561038e+06
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.955442,-0.065218,0.178102,0.135739,2.671709,4.443638,4,5.181242,9.904610e+04,5.517778,-0.436990,-0.127917,1.406565,0.968635,2.379465,1.841727e+06
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.966780,0.000000,0.196227,0.240287,2.692641,17.880575,4,20.780069,-1.223640e+05,72.233011,0.009651,-0.021178,2.561493,0.966231,0.000000,1.604580e+06
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.844453,-0.190647,0.198488,0.150498,2.602846,6.875027,4,2.853935,-1.888807e+03,5.596572,0.013369,0.204565,1.072200,0.989601,1.391732,1.525602e+06
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,0.874423,0.189659,0.185052,0.176741,0.171454,6.114074,15,52.815301,6.979828e+05,4.632054,0.102100,0.554022,1.113012,0.981881,2.562965,1.754212e+06
6,26237,UNIVERSIDADE FEDERAL DE JUIZ DE FORA,0.245185,0.000000,0.647403,0.140351,2.349077,5.880085,4,6.426681,-2.607024e+06,2.696476,0.041165,-0.357271,1.628230,0.800799,0.000000,1.191720e+06
7,26238,UNIVERSIDADE FEDERAL DE MINAS GERAIS,0.931143,-0.177312,0.179566,0.159603,2.598110,9.574437,3,2.178608,1.652545e+05,5.028617,-0.360950,-0.052687,1.501725,1.045176,2.335497,3.252484e+06
8,26239,UNIVERSIDADE FEDERAL DO PARA,0.924645,-0.110699,0.172736,0.188800,1.595068,6.000000,10,2.658700,1.466460e+05,1.674427,0.049172,0.034647,1.232022,0.993761,6.000000,1.506535e+06
9,26240,UNIVERSIDADE FEDERAL DA PARAIBA,0.974350,0.000000,0.167452,0.136707,2.399571,6.215825,4,6.353461,1.052794e+05,5.914929,-0.596215,0.041034,1.169264,1.029583,0.000000,1.957194e+06


In [18]:
# %%
# ==============================================================================
# BLOCO 17: FEATURE F17 - SAZONALIDADE DE DEZEMBRO (FEBRE DO EMPENHO)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F17: Sazonalidade de Dezembro (VPD)")
print("="*50)

import numpy as np
import pandas as pd

# 1. Filtro: Classe 3 (Variações Patrimoniais Diminutivas)
df_vpd_f17 = df[df['Cod_Conta'].str.startswith('3')].copy()

# Garantir que Ano e Mes sejam numéricos para as condicionais de tempo
df_vpd_f17['Ano'] = df_vpd_f17['Ano'].astype(int)
df_vpd_f17['Mes'] = df_vpd_f17['Mes'].astype(int)

# 2. Agrupamento mensal compensatório (Regra de Ouro: mantendo o sinal)
vpd_m_f17 = df_vpd_f17.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_VPD')

# 3. Valor Absoluto após compensação
vpd_m_f17['Valor_VPD'] = vpd_m_f17['Soma_VPD'].abs()

# 4. Função para calcular o multiplicador de dezembro médio (ao longo da série histórica)
def calcular_sazonalidade_f17(group):
    anos = group['Ano'].unique()
    razoes_anuais = []
    
    for ano in anos:
        dados_ano = group[group['Ano'] == ano]
        
        # Isola a média mensal dos meses normais (Janeiro a Novembro)
        media_jan_nov = dados_ano[(dados_ano['Mes'] >= 1) & (dados_ano['Mes'] <= 11)]['Valor_VPD'].mean()
        
        # Isola o valor de Dezembro
        valor_dez = dados_ano[dados_ano['Mes'] == 12]['Valor_VPD'].sum()
        
        # Trava matemática para evitar divisão por zero
        if pd.notna(media_jan_nov) and media_jan_nov > 0:
            razao = valor_dez / media_jan_nov
            razoes_anuais.append(razao)
            
    # Retorna a média dos multiplicadores de dezembro dos 3 anos
    if len(razoes_anuais) > 0:
        return np.mean(razoes_anuais)
    else:
        return 0.0

# 5. Aplicação da métrica por Órgão
f17_resultado = vpd_m_f17.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_sazonalidade_f17).reset_index(name='F17_Sazonalidade_Dez')

# 6. Integração à Matriz Mestre
if 'F17_Sazonalidade_Dez' in df_features.columns:
    df_features = df_features.drop(columns=['F17_Sazonalidade_Dez'])

df_features = pd.merge(df_features, f17_resultado[['Cod_Orgao', 'F17_Sazonalidade_Dez']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F17_Sazonalidade_Dez'] = df_features['F17_Sazonalidade_Dez'].fillna(0)

print("✅ F17 Calculada: 'Síndrome de Dezembro' mapeada com sucesso!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F17_Sazonalidade_Dez']].head())


 Calculando F17: Sazonalidade de Dezembro (VPD)
✅ F17 Calculada: 'Síndrome de Dezembro' mapeada com sucesso!


,Cod_Orgao,Nome_Orgao,F17_Sazonalidade_Dez
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,1.230793
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,1.161474
2,26233,UNIVERSIDADE FEDERAL DO CEARA,1.039698
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,1.183532
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,1.111777


In [19]:
# %%
# ==============================================================================
# BLOCO 18: FEATURE F18 - ÍNDICE DE DESPESAS DE EXERCÍCIOS ANTERIORES (DEA)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F18: Índice DEA (Passivos Ocultos / Elemento 92)")
print("="*50)

import numpy as np
import pandas as pd

# 1. Filtro Mestre: Isolar apenas as Despesas Pagas (Conta 6.2.2.1.3.04...)
df_pagamentos_f18 = df[df['Cod_Conta'].str.startswith('6221304')].copy()

# 2. Agrupamento mensal compensatório (Regra de Ouro) para o TOTAL de Pagamentos
pagamentos_m = df_pagamentos_f18.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_Pagamentos')
pagamentos_m['Valor_Pagamentos'] = pagamentos_m['Soma_Pagamentos'].abs()

# 3. Filtro Específico: Isolar os pagamentos que são DEA (Elemento 92)
df_dea = df_pagamentos_f18[df_pagamentos_f18['Cod_Elemento'] == '92'].copy()

# 4. Agrupamento mensal compensatório (Regra de Ouro) apenas para a DEA
dea_m = df_dea.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index(name='Soma_DEA')
dea_m['Valor_DEA'] = dea_m['Soma_DEA'].abs()

# 5. Consolidar os totais acumulados dos 36 meses por Órgão
# Para calcular a proporção (peso) da DEA sobre todo o período auditado
total_pagamentos_orgao = pagamentos_m.groupby(['Cod_Orgao', 'Nome_Orgao'])['Valor_Pagamentos'].sum().reset_index()
total_dea_orgao = dea_m.groupby(['Cod_Orgao', 'Nome_Orgao'])['Valor_DEA'].sum().reset_index()

# 6. Unir as bases para calcular o Índice
df_f18_calc = pd.merge(total_pagamentos_orgao, total_dea_orgao, on=['Cod_Orgao', 'Nome_Orgao'], how='left').fillna(0)

# 7. Cálculo da Proporção: (DEA / Total de Pagamentos)
# Utilizando np.where para blindar o código contra divisões por zero
df_f18_calc['F18_Indice_DEA'] = np.where(
    df_f18_calc['Valor_Pagamentos'] == 0, 
    0.0, 
    df_f18_calc['Valor_DEA'] / df_f18_calc['Valor_Pagamentos']
)

# 8. Integração à Matriz Mestre
if 'F18_Indice_DEA' in df_features.columns:
    df_features = df_features.drop(columns=['F18_Indice_DEA'])

df_features = pd.merge(df_features, df_f18_calc[['Cod_Orgao', 'F18_Indice_DEA']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F18_Indice_DEA'] = df_features['F18_Indice_DEA'].fillna(0)

print("✅ F18 Calculada com sucesso!")
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F18_Indice_DEA']].head())


 Calculando F18: Índice DEA (Passivos Ocultos / Elemento 92)
✅ F18 Calculada com sucesso!


,Cod_Orgao,Nome_Orgao,F18_Indice_DEA
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.012109
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.019514
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.009041
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.005498
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.005813


In [20]:
# %%
# ==============================================================================
# BLOCO 19: FEATURE F19 - PICO DE CANCELAMENTO DE RAP NP (SAFRA ANUAL)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F19: Pico de Execução Fictícia (Mortalidade de RAP)")
print("="*50)

import numpy as np
import pandas as pd

# 1. Filtros Mapeados na Classe 6 (Execução Orçamentária)
# Usamos uma tupla para capturar tanto o "A Liquidar" (05) quanto o "Em Liquidação" (06)
contas_inscricao = ('6221305', '6221306') 
conta_cancelamento = '6319' # Captura a raiz de todos os RP Não Processados Cancelados

df_inscr_rap = df[df['Cod_Conta'].str.startswith(contas_inscricao)].copy()
df_cancel_rap = df[df['Cod_Conta'].str.startswith(conta_cancelamento)].copy()

# 2. Passo A: Agrupamento Mensal (Regra de Ouro de Compensação)
inscr_m = df_inscr_rap.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index()
cancel_m = df_cancel_rap.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index()

# 3. Passo B: Extração da Magnitude (Valor Absoluto Mensal)
inscr_m['Valor_Inscrito_M'] = inscr_m['Mov_Liquido'].abs()
cancel_m['Valor_Cancelado_M'] = cancel_m['Mov_Liquido'].abs()

# 4. Passo C: Agrupamento Anual (Somando os meses já limpos)
inscr_ano = inscr_m.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano'])['Valor_Inscrito_M'].sum().reset_index(name='Valor_Inscrito')
cancel_ano = cancel_m.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano'])['Valor_Cancelado_M'].sum().reset_index(name='Valor_Cancelado')

# 5. O "Casamento de Safra" (Shifting do Ano)
# A Inscrição do Ano X é a base para o Cancelamento do Ano X+1.
inscr_ano['Ano_Base_Cancelamento'] = inscr_ano['Ano'].astype(int) + 1
cancel_ano['Ano'] = cancel_ano['Ano'].astype(int)

# 6. Cruzamento exato (Ano a Ano)
df_f19_anual = pd.merge(
    cancel_ano, 
    inscr_ano[['Cod_Orgao', 'Ano_Base_Cancelamento', 'Valor_Inscrito']], 
    left_on=['Cod_Orgao', 'Ano'], 
    right_on=['Cod_Orgao', 'Ano_Base_Cancelamento'], 
    how='left'
).fillna(0)

# 7. Cálculo da Taxa Anual de Cancelamento
df_f19_anual['Taxa_Anual_Cancel'] = np.where(
    df_f19_anual['Valor_Inscrito'] == 0, 
    0.0, 
    df_f19_anual['Valor_Cancelado'] / df_f19_anual['Valor_Inscrito']
)

# 8. Extração do Pico Máximo para o Modelo (O Pior Ano da Gestão)
f19_resultado = df_f19_anual.groupby(['Cod_Orgao', 'Nome_Orgao'])['Taxa_Anual_Cancel'].max().reset_index(name='F19_Max_Cancel_RAP')

# 9. Integração à Matriz Mestre
if 'F19_Max_Cancel_RAP' in df_features.columns:
    df_features = df_features.drop(columns=['F19_Max_Cancel_RAP'])

df_features = pd.merge(df_features, f19_resultado[['Cod_Orgao', 'F19_Max_Cancel_RAP']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F19_Max_Cancel_RAP'] = df_features['F19_Max_Cancel_RAP'].fillna(0)

print("✅ F19 Calculada: Taxa de execução orçamentária fictícia mapeada!")

# Exibindo o topo do ranking de cancelamentos (Ajustando casas decimais para visualizar melhor as taxas)
pd.options.display.float_format = '{:,.4f}'.format
display(df_features[['Cod_Orgao', 'Nome_Orgao', 'F19_Max_Cancel_RAP']].head(10))


 Calculando F19: Pico de Execução Fictícia (Mortalidade de RAP)
✅ F19 Calculada: Taxa de execução orçamentária fictícia mapeada!


,Cod_Orgao,Nome_Orgao,F19_Max_Cancel_RAP
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.1995
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.0701
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.0967
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.1839
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.0432
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,0.0922
6,26237,UNIVERSIDADE FEDERAL DE JUIZ DE FORA,0.0537
7,26238,UNIVERSIDADE FEDERAL DE MINAS GERAIS,0.0165
8,26239,UNIVERSIDADE FEDERAL DO PARA,0.1458
9,26240,UNIVERSIDADE FEDERAL DA PARAIBA,0.3248


In [21]:
# %%
# ==============================================================================
# BLOCO 20: FEATURE F20 - VOLATILIDADE DE PROVISÕES (BIG BATH ACCOUNTING)
# ==============================================================================
print("\n" + "="*50)
print(" Calculando F20: Volatilidade de Provisões (Contas 2.1.7 e 2.2.7)")
print("="*50)

import numpy as np
import pandas as pd

# 1. Filtro: Isolando o lado do Passivo (Curto e Longo Prazo) com a base do PCASP atualizada
contas_provisao = ('217', '227') 
df_provisoes = df[df['Cod_Conta'].str.startswith(contas_provisao)].copy()

# 2. Agrupamento mensal compensatório (Regra de Ouro)
prov_m = df_provisoes.groupby(['Cod_Orgao', 'Nome_Orgao', 'Ano', 'Mes'])['Mov_Liquido'].sum().reset_index()

# 3. Extraindo a magnitude financeira absoluta
prov_m['Valor_Provisao'] = prov_m['Mov_Liquido'].abs()

# 4. Função do Coeficiente de Variação (CV)
def calcular_cv_f20(group):
    valores = group['Valor_Provisao'].values
    
    if len(valores) < 2 or np.mean(valores) == 0:
        return 0.0
    
    desvio_padrao = np.std(valores, ddof=1)
    media = np.mean(valores)
    
    return desvio_padrao / media

# 5. Aplicação da métrica por Órgão (Análise do Triênio)
f20_resultado = prov_m.groupby(['Cod_Orgao', 'Nome_Orgao']).apply(calcular_cv_f20).reset_index(name='F20_Volatilidade_Provisoes')

# 6. Integração Final à Matriz Mestre
if 'F20_Volatilidade_Provisoes' in df_features.columns:
    df_features = df_features.drop(columns=['F20_Volatilidade_Provisoes'])

df_features = pd.merge(df_features, f20_resultado[['Cod_Orgao', 'F20_Volatilidade_Provisoes']], 
                       on='Cod_Orgao', 
                       how='left')

df_features['F20_Volatilidade_Provisoes'] = df_features['F20_Volatilidade_Provisoes'].fillna(0)

print("✅ F20 Calculada: Saltos absurdos de passivo oculto rastreados com sucesso!")

# Exibindo o Top 10 com as maiores volatilidades
pd.options.display.float_format = '{:,.2f}'.format
display(df_features)


 Calculando F20: Volatilidade de Provisões (Contas 2.1.7 e 2.2.7)
✅ F20 Calculada: Saltos absurdos de passivo oculto rastreados com sucesso!


,Cod_Orgao,Nome_Orgao,F1_Cor_VPA_RecArr,F2_Cor_VPA_Cred,F3_Volatilidade_VPA,F4_Volatilidade_VPD,F5_Var_Pag_Repasse,F6_Var_RPE_PL,F7_Qtd_Deficit_Fin,F8_Var_Margem_Pat,...,F11_Ang_Passivo,F12_Cor_FCO_RPE,F13_Var_Inv_Custeio,F14_Var_Disp_Caixa,F15_Var_Creditos,F16_Ang_VPD,F17_Sazonalidade_Dez,F18_Indice_DEA,F19_Max_Cancel_RAP,F20_Volatilidade_Provisoes
0,26231,UNIVERSIDADE FEDERAL DE ALAGOAS,0.88,0.00,0.20,0.15,2.73,4.60,3,2.19,...,0.04,0.08,2.07,0.88,0.00,"1,364,657.47",1.23,0.01,0.20,0.00
1,26232,UNIVERSIDADE FEDERAL DA BAHIA,0.97,0.00,0.18,0.20,2.63,6.27,3,4.89,...,-0.01,-0.05,1.33,0.92,0.00,"2,561,037.95",1.16,0.02,0.07,0.00
2,26233,UNIVERSIDADE FEDERAL DO CEARA,0.96,-0.07,0.18,0.14,2.67,4.44,4,5.18,...,-0.44,-0.13,1.41,0.97,2.38,"1,841,726.99",1.04,0.01,0.10,0.00
3,26234,UNIVERSIDADE FEDERAL DO ESPIRITO SANTO,0.97,0.00,0.20,0.24,2.69,17.88,4,20.78,...,0.01,-0.02,2.56,0.97,0.00,"1,604,580.10",1.18,0.01,0.18,0.00
4,26235,UNIVERSIDADE FEDERAL DE GOIAS,0.84,-0.19,0.20,0.15,2.60,6.88,4,2.85,...,0.01,0.20,1.07,0.99,1.39,"1,525,602.45",1.11,0.01,0.04,0.00
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,0.87,0.19,0.19,0.18,0.17,6.11,15,52.82,...,0.10,0.55,1.11,0.98,2.56,"1,754,212.33",0.82,0.01,0.09,0.00
6,26237,UNIVERSIDADE FEDERAL DE JUIZ DE FORA,0.25,0.00,0.65,0.14,2.35,5.88,4,6.43,...,0.04,-0.36,1.63,0.80,0.00,"1,191,719.68",1.06,0.00,0.05,0.00
7,26238,UNIVERSIDADE FEDERAL DE MINAS GERAIS,0.93,-0.18,0.18,0.16,2.60,9.57,3,2.18,...,-0.36,-0.05,1.50,1.05,2.34,"3,252,483.90",1.30,0.00,0.02,0.00
8,26239,UNIVERSIDADE FEDERAL DO PARA,0.92,-0.11,0.17,0.19,1.60,6.00,10,2.66,...,0.05,0.03,1.23,0.99,6.00,"1,506,534.82",0.71,0.00,0.15,0.00
9,26240,UNIVERSIDADE FEDERAL DA PARAIBA,0.97,0.00,0.17,0.14,2.40,6.22,4,6.35,...,-0.60,0.04,1.17,1.03,0.00,"1,957,193.71",1.07,0.00,0.32,0.00


In [26]:
# Exporta a base consolidada de features para a pasta de arquivos tabulares
df_features.to_csv('arquivos_tabulares/df_features_consolidadas.csv', sep=';', decimal=',', index=False)

In [22]:
# %%
# ==============================================================================
# BLOCO 21: SANITY CHECK - AUDITORIA FINAL DA MATRIZ DE FEATURES
# ==============================================================================
print("\n" + "="*50)
print(" Raio-X da Matriz: Preparando para o Isolation Forest")
print("="*50)

import pandas as pd
import numpy as np

# 1. Visão Geral da Matriz
print(f"\n📊 Formato da Matriz Final: {df_features.shape[0]} Órgãos e {df_features.shape[1]} Colunas (Identificadores + Features)")

# 2. Verificação de Valores Nulos (NaN)
nulos = df_features.isna().sum()
colunas_com_nulos = nulos[nulos > 0]

print("\n⚠️ Verificação de Nulos (NaN):")
if colunas_com_nulos.empty:
    print("✅ Excelente! Nenhum valor nulo (NaN) encontrado na matriz.")
else:
    print("❌ Atenção! Encontramos valores nulos nas seguintes features:")
    print(colunas_com_nulos)

# 3. Verificação de Valores Infinitos (Inf) 
# Seleciona apenas as colunas numéricas para não testar as strings (Nome do Órgão)
cols_numericas = df_features.select_dtypes(include=[np.number]).columns
infinitos = np.isinf(df_features[cols_numericas]).sum()
colunas_com_inf = infinitos[infinitos > 0]

print("\n⚠️ Verificação de Infinitos (Inf):")
if colunas_com_inf.empty:
    print("✅ Perfeito! Nenhum valor infinito encontrado. As travas de divisão por zero funcionaram.")
else:
    print("❌ Cuidado! Valores infinitos encontrados:")
    print(colunas_com_inf)

# ==============================================================================
# 4. A VACINA: Tratamento Final de Garantia
# Mesmo que tenha dado tudo certo acima, rodamos isso para blindar o df_features
# Transforma qualquer Inf escondido em NaN, e depois transforma todo NaN em 0.0
# ==============================================================================
df_features.replace([np.inf, -np.inf], np.nan, inplace=True)
df_features.fillna(0.0, inplace=True)

# 5. Resumo Estatístico
print("\n📈 Resumo Estatístico Final (Amostra do comportamento dos dados):")
pd.options.display.float_format = '{:,.4f}'.format

# Mostra o count, mean, std, min e max de todas as 20 features
display(df_features.describe().T)


 Raio-X da Matriz: Preparando para o Isolation Forest

📊 Formato da Matriz Final: 45 Órgãos e 22 Colunas (Identificadores + Features)

⚠️ Verificação de Nulos (NaN):
✅ Excelente! Nenhum valor nulo (NaN) encontrado na matriz.

⚠️ Verificação de Infinitos (Inf):
✅ Perfeito! Nenhum valor infinito encontrado. As travas de divisão por zero funcionaram.

📈 Resumo Estatístico Final (Amostra do comportamento dos dados):


,count,mean,std,min,25%,50%,75%,max
F1_Cor_VPA_RecArr,45.0000,0.7569,0.3112,-0.1105,0.7205,0.8994,0.9554,0.9948
F2_Cor_VPA_Cred,45.0000,-0.0084,0.0517,-0.1906,0.0000,0.0000,0.0000,0.1897
F3_Volatilidade_VPA,45.0000,0.3090,0.2221,0.1552,0.1833,0.2121,0.3163,1.0944
F4_Volatilidade_VPD,45.0000,0.2190,0.1239,0.1226,0.1519,0.1830,0.2403,0.7773
F5_Var_Pag_Repasse,45.0000,2.2049,0.6924,0.1715,2.1298,2.4814,2.6278,2.8362
F6_Var_RPE_PL,45.0000,10.2082,20.1743,2.7102,4.6019,6.0000,6.9658,138.2169
F7_Qtd_Deficit_Fin,45.0000,5.8444,3.8727,3.0000,3.0000,4.0000,8.0000,19.0000
F8_Var_Margem_Pat,45.0000,13.6695,29.2171,1.4503,3.9417,6.0386,11.0693,192.2879
F9_Ang_RPE,45.0000,"-154,619.0276","819,366.6292","-4,203,014.4101","-94,842.2475","35,490.1318","145,715.7027","697,982.7973"
F10_Volatilidade_NFCP,45.0000,8.2072,12.1091,1.4008,3.2442,5.0286,6.5228,72.2330


In [24]:
# ==============================================================================
# BLOCO PTT: MOTOR HEURÍSTICO (SISTEMA DE ESCORAGEM POR REGRAS)
# ==============================================================================
print("\n" + "="*50)
print(" Gerando Ranking Heurístico para o PTT (Regras Especialistas)")
print("="*50)

import pandas as pd
import numpy as np

# 1. Cópia da Matriz Mestre
df_ranking = df_features.copy()
colunas_flags = []

# 2. Definindo a "Zona de Risco" (Piores 5% da distribuição)
LIMITE_RISCO = 0.95 

print("🚩 Aplicando radares de anomalia nas 20 dimensões...\n")

for col in df_features.columns:
    if col.startswith('F'):
        nome_flag = f'Flag_{col.split("_")[0]}'
        colunas_flags.append(nome_flag)
        
        # Tratamento especial para a F16 (Onde o risco é ser o MENOR valor - desidratação)
        if col == 'F16_Ang_VPD':
            linha_de_corte = df_ranking[col].quantile(1 - LIMITE_RISCO) # Pega os 5% menores
            df_ranking[nome_flag] = np.where(df_ranking[col] <= linha_de_corte, 1, 0)
            
        # Tratamento para as demais (Onde o risco é ser o MAIOR valor - alta volatilidade/correlação)
        else:
            linha_de_corte = df_ranking[col].quantile(LIMITE_RISCO) # Pega os 5% maiores
            df_ranking[nome_flag] = np.where(df_ranking[col] >= linha_de_corte, 1, 0)

# 3. Calculando o Score Total (Quantas Red Flags a universidade acionou?)
df_ranking['Score_Risco_Total'] = df_ranking[colunas_flags].sum(axis=1)

# 4. Isolando o Top 10 das Unidades mais Críticas
ranking_final = df_ranking[['Cod_Orgao', 'Nome_Orgao', 'Score_Risco_Total'] + colunas_flags]
ranking_final = ranking_final.sort_values(by='Score_Risco_Total', ascending=False)

print("✅ Ranking Heurístico Concluído!")
print("Abaixo, as Universidades que dispararam o maior número de Red Flags simultâneas:")
print("Limite de Risco: As 5% piores valores em cada feature foram considerados como zona de risco (1 ponto).")

# Exibindo o resultado para o relatório do PTT
display(ranking_final[['Cod_Orgao', 'Nome_Orgao', 'Score_Risco_Total']].head(15))


 Gerando Ranking Heurístico para o PTT (Regras Especialistas)
🚩 Aplicando radares de anomalia nas 20 dimensões...

✅ Ranking Heurístico Concluído!
Abaixo, as Universidades que dispararam o maior número de Red Flags simultâneas:
Limite de Risco: As 5% piores valores em cada feature foram considerados como zona de risco (1 ponto).


,Cod_Orgao,Nome_Orgao,Score_Risco_Total
5,26236,UNIVERSIDADE FEDERAL FLUMINENSE,7
43,26456,UNIVERSIDADE FEDERAL DO AGRESTE DE PERNAMBUCO,7
18,26249,UNIVERSIDADE FEDERAL RURAL DO RIO DE JANEIRO,7
38,26450,UNIVERSIDADE FEDERAL DO SUL DA BAHIA,6
30,26269,UNIVERSIDADE FEDERAL DO ESTADO RIO DE JANEIRO,6
44,26457,UNIVERSIDADE FEDERAL DO NORTE DE TOCANTINS,6
34,26442,UNIV.DA INTEG.INTERN.DA LUSOF.AFRO-BRASILEIRA,5
19,26252,UNIVERSIDADE FEDERAL DE CAMPINA GRANDE,5
29,26267,UNIVERS. FEDERAL DA INTEG. LATINO AMERICANA,5
42,26455,UNIVERSIDADE FEDERAL DO DELTA DO PARNAIBA,4
